# Replit ↔ Colab Bridge — Backend Server

**Запустите все ячейки по порядку (Runtime → Run all).**

После запуска последней ячейки скопируйте ngrok URL и вставьте его в поле CONNECTION в интерфейсе Replit.

---
**Что делает этот ноутбук:**
1. Устанавливает зависимости (Flask, ngrok, openai)
2. Запрашивает ваш OpenRouter API ключ
3. Запускает Flask-сервер на порту 5000
4. Открывает публичный туннель через ngrok
5. Выводит URL, который нужно скопировать в Replit

In [ ]:
# ============================================================
# Ячейка 1: Установка зависимостей
# ============================================================
!pip install -q flask flask-cors pyngrok openai requests

In [ ]:
# ============================================================
# Ячейка 2: Настройка — введите ваши ключи
# ============================================================
import getpass
import os

# Ключ OpenRouter (получить на openrouter.ai/keys)
OPENROUTER_API_KEY = getpass.getpass('Введите ваш OpenRouter API ключ: ')

# Токен ngrok (получить на dashboard.ngrok.com/get-started/your-authtoken)
# Без токена сессия ограничена 2 часами
NGROK_TOKEN = getpass.getpass('Введите ngrok authtoken (или Enter чтобы пропустить): ')

print('✓ Ключи получены')

In [ ]:
# ============================================================
# Ячейка 3: Flask-сервер (бэкенд)
# ============================================================
import threading
import time
import json
import requests as req_lib
from flask import Flask, request, jsonify
from flask_cors import CORS
from openai import OpenAI

app = Flask(__name__)
CORS(app)

openrouter_client = OpenAI(
    base_url='https://openrouter.ai/api/v1',
    api_key=OPENROUTER_API_KEY,
)

# ----------------------------------------------------------
# Health check
# ----------------------------------------------------------
@app.route('/health', methods=['GET'])
def health():
    return jsonify({'status': 'ok', 'version': '2.0'})


# ----------------------------------------------------------
# Список бесплатных моделей
# ----------------------------------------------------------
@app.route('/models/free', methods=['GET'])
def get_free_models():
    try:
        resp = req_lib.get(
            'https://openrouter.ai/api/v1/models',
            headers={'Authorization': f'Bearer {OPENROUTER_API_KEY}'},
            timeout=10
        )
        resp.raise_for_status()
        all_models = resp.json().get('data', [])

        free_models = []
        for m in all_models:
            pricing = m.get('pricing', {})
            prompt_price = str(pricing.get('prompt', '1'))
            completion_price = str(pricing.get('completion', '1'))
            is_free = (prompt_price in ('0', '0.0', '') and
                       completion_price in ('0', '0.0', '')) or \
                       m.get('id', '').endswith(':free')
            if is_free:
                top_provider = m.get('top_provider', {})
                free_models.append({
                    'id': m.get('id', ''),
                    'name': m.get('name', m.get('id', '')),
                    'description': m.get('description', ''),
                    'context_length': m.get('context_length', 0),
                    'pricing': pricing,
                    'top_provider': top_provider.get('context_length', m.get('id', '').split('/')[0] if '/' in m.get('id', '') else ''),
                    'average_rating': None,
                })

        return jsonify({'status': 'ok', 'models': free_models, 'count': len(free_models)})
    except Exception as e:
        return jsonify({'status': 'error', 'message': str(e), 'models': [], 'count': 0}), 500


# ----------------------------------------------------------
# Запуск эксперимента
# ----------------------------------------------------------
def fetch_model_card(model_id):
    """Получить карточку модели из OpenRouter."""
    try:
        resp = req_lib.get(
            f'https://openrouter.ai/api/v1/models/{model_id}',
            headers={'Authorization': f'Bearer {OPENROUTER_API_KEY}'},
            timeout=5
        )
        if resp.status_code == 200:
            m = resp.json()
            pricing = m.get('pricing', {})
            return {
                'id': m.get('id', model_id),
                'name': m.get('name', model_id),
                'description': m.get('description', ''),
                'context_length': m.get('context_length', None),
                'pricing': pricing,
                'top_provider': (m.get('top_provider') or {}).get('context_length', model_id.split('/')[0] if '/' in model_id else ''),
                'average_rating': None,
            }
    except Exception:
        pass
    return {'id': model_id, 'name': model_id, 'context_length': None, 'top_provider': None, 'average_rating': None}


def run_single(model_id, messages, temperature, max_tokens):
    """Выполнить один запрос к OpenRouter."""
    kwargs = {
        'messages': messages,
        'temperature': temperature,
        'max_tokens': max_tokens,
    }
    if model_id:
        kwargs['model'] = model_id
    else:
        # Автовыбор — OpenRouter выберет свободную модель
        kwargs['model'] = 'openrouter/auto'

    t_start = time.time()
    response = openrouter_client.chat.completions.create(**kwargs)
    elapsed = round(time.time() - t_start, 3)

    result_text = response.choices[0].message.content or ''
    model_used = response.model or model_id or 'unknown'
    usage = response.usage

    return {
        'result': result_text,
        'model': model_used,
        'metrics': {
            'prompt_tokens': usage.prompt_tokens if usage else 0,
            'completion_tokens': usage.completion_tokens if usage else 0,
            'total_tokens': usage.total_tokens if usage else 0,
            'execution_time': elapsed,
        }
    }


@app.route('/run', methods=['POST'])
def run_experiment():
    data = request.get_json()
    if not data:
        return jsonify({'status': 'error', 'message': 'No JSON body'}), 400

    task_type = data.get('task_type', 'text_generation')
    mode = data.get('mode', 'auto')
    model_id = data.get('model')  # None = auto
    models_list = data.get('models', [])
    params = data.get('params', {})

    prompt = params.get('prompt', '')
    temperature = float(params.get('temperature', 0.7))
    max_tokens = int(params.get('max_tokens', 1024))
    system_prompt = params.get('system_prompt')
    custom_code = params.get('custom_code')

    messages = []
    if system_prompt:
        messages.append({'role': 'system', 'content': system_prompt})
    messages.append({'role': 'user', 'content': prompt})

    try:
        # Сравнение моделей
        if task_type == 'model_comparison' and models_list:
            comparisons = []
            for mid in models_list:
                try:
                    r = run_single(mid, messages, temperature, max_tokens)
                    comparisons.append({
                        'model_id': mid,
                        'result': r['result'],
                        'prompt_tokens': r['metrics']['prompt_tokens'],
                        'completion_tokens': r['metrics']['completion_tokens'],
                        'total_tokens': r['metrics']['total_tokens'],
                        'execution_time': r['metrics']['execution_time'],
                        'error': None,
                    })
                except Exception as e:
                    comparisons.append({'model_id': mid, 'result': '', 'error': str(e)})
            return jsonify({'status': 'ok', 'task_type': task_type, 'mode': mode, 'comparisons': comparisons})

        # Кастомный код
        if task_type == 'custom_code' and custom_code:
            import io, contextlib
            namespace = {'openrouter_client': openrouter_client, 'OPENROUTER_API_KEY': OPENROUTER_API_KEY}
            stdout_capture = io.StringIO()
            t_start = time.time()
            with contextlib.redirect_stdout(stdout_capture):
                exec(custom_code, namespace)
            elapsed = round(time.time() - t_start, 3)
            output = stdout_capture.getvalue()
            return jsonify({
                'status': 'ok', 'task_type': task_type, 'mode': mode,
                'result': output,
                'metrics': {'prompt_tokens': 0, 'completion_tokens': 0, 'total_tokens': 0, 'execution_time': elapsed},
            })

        # Обычная генерация текста / system prompt
        r = run_single(model_id, messages, temperature, max_tokens)
        model_used = r['model']
        model_card = fetch_model_card(model_used)

        return jsonify({
            'status': 'ok',
            'task_type': task_type,
            'mode': mode,
            'result': r['result'],
            'model': model_used,
            'model_card': model_card,
            'metrics': r['metrics'],
        })

    except Exception as e:
        return jsonify({'status': 'error', 'task_type': task_type, 'mode': mode, 'message': str(e)}), 500


print('✓ Flask-сервер определён')

In [ ]:
# ============================================================
# Ячейка 4: Запуск сервера + ngrok туннель
# ============================================================
from pyngrok import ngrok, conf

# Настройка токена ngrok (если предоставлен)
if NGROK_TOKEN and NGROK_TOKEN.strip():
    ngrok.set_auth_token(NGROK_TOKEN.strip())
    print('✓ ngrok токен установлен')

# Запускаем Flask в отдельном потоке
flask_thread = threading.Thread(
    target=lambda: app.run(host='0.0.0.0', port=5000, use_reloader=False, debug=False)
)
flask_thread.daemon = True
flask_thread.start()

time.sleep(2)  # Даём серверу стартовать

# Открываем туннель
public_url = ngrok.connect(5000)
tunnel_url = public_url.public_url

print('\n' + '='*60)
print('  СЕРВЕР ЗАПУЩЕН')
print('='*60)
print(f'\n  URL для Replit:')
print(f'\n  >>> {tunnel_url} <<<')
print(f'\n  Скопируйте этот URL в поле CONNECTION в интерфейсе Replit')
print(f'  и нажмите кнопку PING для проверки соединения.')
print('\n' + '='*60)
print('  Сервер активен. Не закрывайте эту вкладку!')
print('='*60)

## Доступные эндпоинты

| Метод | Путь | Описание |
|-------|------|----------|
| GET | `/health` | Проверка статуса сервера |
| GET | `/models/free` | Список бесплатных моделей OpenRouter |
| POST | `/run` | Запуск эксперимента |

## Типы заданий (`task_type`)

- `text_generation` — обычная генерация текста
- `model_comparison` — сравнение нескольких моделей на одном промпте
- `system_prompt` — генерация с системным промптом
- `custom_code` — выполнение произвольного Python-кода

## Важные замечания

- Сессия Colab завершается автоматически через ~12 часов (бесплатный план)
- Без ngrok-токена туннель живёт ~2 часа
- Не закрывайте вкладку Colab пока работаете с Replit
- При истечении сессии — перезапустите все ячейки и скопируйте новый URL